# SketchyGAN on Google Colab

This notebook runs the replicated SketchyGAN code in Colab using TF1 compatibility mode.

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/yashdive/sketch-Image.git
%cd /content/sketch-Image
!git checkout cursor/replicate-sketchygan-notebook-0c55

In [ ]:
!pip -q install --upgrade pip
!pip -q install "tensorflow==2.15.0" tf_slim numpy scipy opencv-python

In [ ]:
from pathlib import Path

patched = []
for p in Path('.').rglob('*.py'):
    if '.git' in p.parts:
        continue
    txt = p.read_text(encoding='utf-8')
    new = txt

    if 'import tensorflow as tf' in new:
        new = new.replace(
            'import tensorflow as tf',
            'import tensorflow.compat.v1 as tf\ntf.disable_v2_behavior()'
        )

    if 'slim = tf.contrib.slim' in new:
        new = new.replace('slim = tf.contrib.slim', 'import tf_slim as slim')

    if new != txt:
        p.write_text(new, encoding='utf-8')
        patched.append(str(p))

print(f'Patched {len(patched)} files')
for f in patched[:20]:
    print(' -', f)

In [ ]:
!python - <<'PY'
import main_single
print('main_single imported successfully')
PY

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Set these two paths to your Drive folders containing TFRecord files before running the next cell.

In [ ]:
SKETCHY_DRIVE_DIR = '/content/drive/MyDrive/SketchyGAN/training_data/sketchy'
FLICKR_DRIVE_DIR = '/content/drive/MyDrive/SketchyGAN/training_data/flickr_output'

In [ ]:
!mkdir -p /content/training_data
!ln -sfn "$SKETCHY_DRIVE_DIR" /content/training_data/sketchy
!ln -sfn "$FLICKR_DRIVE_DIR" /content/training_data/flickr_output
!echo "Sketchy link -> $(readlink -f /content/training_data/sketchy)"
!echo "Flickr link  -> $(readlink -f /content/training_data/flickr_output)"

In [ ]:
!mkdir -p inception_v4_model
!wget -q http://download.tensorflow.org/models/inception_v4_2016_09_09.tar.gz -O /content/inception_v4_2016_09_09.tar.gz
!tar -xzf /content/inception_v4_2016_09_09.tar.gz -C inception_v4_model
!ls -lh inception_v4_model | sed -n '1,20p'

In [ ]:
!python main_single.py --mode train --batch_size 2 --num_gpu 1 --small_img 1 --distance_map 1 --max_iter_step 10000

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/sketch-Image --port 6006

In [ ]:
# Replace resume_from with your checkpoint suffix from ckpt_skgan_<timestamp>
!python main_single.py --mode test --resume_from "YYYY-MM-DD-HH-MM-SS" --num_gpu 1